# QUBO feature selection (Romero et al. 2025)

Reimplementation of Eq. (4) from Romero, Gupta, Gatlin, Chapkin, Cai,
*Quantum Machine Intelligence* (2025) 7:114, applied to GEO **GSE308682**.

Logic lives in `qubo_model.py`, `data_loader.py`, `qubo_experiment.py`. This notebook only configures and runs the experiment.

Comparison vs the paper: [`docs/compare-romero-2025.html`](docs/compare-romero-2025.html).


In [ ]:
# Core deps. Optional QBoson: pip install kaiwu==1.3.1 torch
# pip install git+https://github.com/qboson/kaiwu-pytorch-plugin.git
%pip install -q numpy scikit-learn matplotlib seaborn dwave-ocean-sdk scanpy ipywidgets tqdm


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "qubo_model.py").exists():
    ROOT = Path("/Users/jinshang/Projects/qubo")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from qubo_model import (
    HAS_DWAVE,
    HAS_KAIWU,
    HAS_KAIWU_PLUGIN,
    HAS_TABU,
    available_solvers,
    compute_mutual_information_matrix,
    solve_qubo_target_k,
    solver_banner,
)
from qubo_experiment import (
    compare_with_lasso_rfr,
    load_experiment,
    print_regression_mse,
    target_cardinality,
)

print(f"Importing Python modules from {ROOT}")
print(f"Ocean tabu={HAS_TABU}, Ocean SA={HAS_DWAVE}, Kaiwu SDK={HAS_KAIWU}, torch plugin={HAS_KAIWU_PLUGIN}")


## Solvers

`dwave-ocean-sdk` is already used. **`tabu` / `sa` are classical CPU samplers**, not a QPU.
`leap` is D-Wave’s cloud hybrid solver. `kaiwu_cim` is QBoson’s photonic CIM.

| `SOLVER` | Stack | Hardware | Env |
| --- | --- | --- | --- |
| `tabu` | Ocean | classical CPU | — |
| `sa` | Ocean | classical CPU | — |
| `leap` | Ocean | Leap hybrid | `DWAVE_API_TOKEN` |
| `custom_sa` | this repo | classical CPU | — |
| `kaiwu_sa` / `kaiwu_tabu` | kaiwu SDK | classical CPU | optional license |
| `kaiwu_cim` | kaiwu CIM | photonic QPU | `KAIWU_USER_ID`, `KAIWU_SDK_CODE` |


In [ ]:
# Romero et al. §4.1-style settings. Implementations: data_loader.py, qubo_model.py
# T: "pseudotime" = scanpy DPT. "gene" = held-out RUNX1 Pearson residual.
# N_TOP_GENES: paper used ~5,000 HVGs. Pairwise MI is O(p²); tqdm shows progress.
USE_REAL_DATA = True
N_TOP_GENES = 5000
TARGET_MODE = "pseudotime"
ROOT_GENE = "HBE1"          # DPT iroot = cell with min residual of this gene
SOLVER = "tabu"             # tabu | sa | leap | custom_sa | kaiwu_sa | kaiwu_tabu | kaiwu_cim
K_TARGET = 50               # paper real-data cardinality

print(solver_banner(SOLVER))
print("Available solvers:")
for name, meta in available_solvers().items():
    mark = "yes" if meta["installed"] else "no"
    print(f"  {name:12} installed={mark:3}  {meta['kind']:16}  {meta['stack']}")

X, y, true_features, feature_names = load_experiment(
    use_real_data=USE_REAL_DATA,
    n_top_genes=N_TOP_GENES,
    target_mode=TARGET_MODE,
    root_gene=ROOT_GENE,
)


In [ ]:
# Discrete MI: qubo_model.compute_mutual_information_matrix (tqdm on bins, I, R)
I, R = compute_mutual_information_matrix(X, y, n_bins=10)
print(f"Importance I (first 5): {I[:5]}")
print(f"Redundancy R (5x5):\n{R[:5, :5]}")


In [ ]:
K = target_cardinality(X.shape[1], k=K_TARGET)
print(f"Target cardinality K={K}")


In [ ]:
selected_features_qubo, energy, alpha, Q = solve_qubo_target_k(
    I, R, k=K, solver=SOLVER
)
selected_idx_qubo = np.where(selected_features_qubo == 1)[0]
print(f"α*={alpha:.4f}, energy={energy:.4f}, |F*|={len(selected_idx_qubo)}")
print(f"QUBO selected indices: {selected_idx_qubo}")
if feature_names is not None:
    print("QUBO selected genes:", [feature_names[i] for i in selected_idx_qubo])

if len(selected_idx_qubo) != K:
    print(
        f"WARNING: |F*|={len(selected_idx_qubo)} != target K={K}. "
        "α-search/solver did not hit paper cardinality. "
        "LASSO/RF below are matched to |F*|, not k=50. "
        "Do not treat this as a Romero et al. k=50 gene list."
    )
k_compare = max(len(selected_idx_qubo), 1)


In [ ]:
# LASSO path (~K nonzeros) and random forest. Overlaps printed here.
selected_idx_lasso, selected_idx_rf = compare_with_lasso_rfr(
    X, y, I, true_features, selected_idx_qubo, K=k_compare, feature_names=feature_names
)


In [ ]:
print_regression_mse(X, y, selected_idx_qubo, selected_idx_lasso, selected_idx_rf)


## After a run

Open [`docs/compare-romero-2025.html`](docs/compare-romero-2025.html) for the last analyzed comparison vs the paper.

A healthy unconstrained Eq. (4) solve has **negative energy** and **|F\*| ≈ K** (50). Large positive energy with |F\*| ≈ p/2 means tabu/α-search failed to find a sparse minimizer — switch solver (`sa`, `leap`, `kaiwu_cim`) or inspect the α bisection tqdm postfix (`n_sel`).
